### 5) 데이터 분리

In [33]:
import pandas as pd

#   기간에 따른 토크나이징 적용 결과 파일 불러오기
df_tokenizing_01 = pd.read_pickle('./sentiment_tokenized_dataset_260101_260430.pkl')
df_tokenizing_02 = pd.read_pickle('./sentiment_tokenized_dataset_260501_260523.pkl')

In [34]:
#   각 파일 df_tokenizing로 병합
df_tokenizing = pd.concat([df_tokenizing_01, df_tokenizing_02], axis = 0, ignore_index = True)

len(df_tokenizing_01),len(df_tokenizing_02),len(df_tokenizing)

(524806, 89237, 614043)

In [35]:
#   병합 후 열별 결측 확인
for col in df_tokenizing.columns.tolist():
    print(df_tokenizing.isnull()[col].value_counts())

"""경제뉴스는 2026년 05월 23일까지 수집되었으나
   증권 데이터는 2026년 05월 21일까지 수집되어 결측이 발생했습니다."""

#   'Target_Date' 열 결측 부분 제거
df_tokenizing.dropna(subset = ['Target_Date'], inplace=True)

#   'Target_Date' 열 결측 값 재확인
print('Target_Date 결측 값: ', df_tokenizing['Target_Date'].isnull().sum(), '개')

섹션
False    614043
Name: count, dtype: int64
제목
False    614043
Name: count, dtype: int64
언론사
False    614043
Name: count, dtype: int64
본문
False    614043
Name: count, dtype: int64
Target_Date
False    608806
True       5237
Name: count, dtype: int64
answer
False    608806
True       5237
Name: count, dtype: int64
수정
False    614043
Name: count, dtype: int64
tokens
False    614043
Name: count, dtype: int64
Target_Date 결측 값:  0 개


In [36]:
df_tokenizing.head()

,섹션,제목,언론사,본문,Target_Date,answer,수정,tokens
0,258,"잘 나가는 방위산업株…한화에어로·LIG넥스원, 나란히 AA로 신용 ‘레벨업’ [투자...",헤럴드경제,"2025년 활약한 방위산업株, 신용평가 등급 A+~AA 포진 한화에어로·LIG넥스원...",2026-01-02,True,2025년 활약한 방위산업 신용평가 등급 A AA 포진 한화에어로 LIG넥스원...,"[활약, 방위, 산업, 신용, 평가, 등급, 포진, 한화에어로, 넥스원, 상향, 증..."
1,263,오세훈 서울시장 “비상계엄 등 잘못 인정하고 반성해야”,이코노미스트,국민의힘 계엄 반성 등 과거와의 단절 요구 ‘국민이 먹고 사는 문제’ 집중해야 한다...,2026-01-02,True,국민의힘 계엄 반성 등 과거와의 단절 요구 국민이 먹고 사는 문제 집중해야 한다...,"[국민, 힘, 계엄, 반성, 과거, 단절, 요구, 국민, 먹, 사, 문제, 집중, ..."
2,263,전기차 국고보조금 작년과 동일…내연차 폐차·매각후 전기차 사면 100만원 더,부산일보,"기후부, 전기차 구매보조금 개편안 발표 5300만원 미만 차 보조금 100% 지급 ...",2026-01-02,True,기후부 전기차 구매보조금 개편안 발표 5300만원 미만 차 보조금 100 지급 ...,"[기후, 전기, 차, 구매, 보조금, 개편안, 발표, 미만, 차, 보조금, 지, 급..."
3,263,새해 첫날 일부 복권판매점서 '로또발행 일시 중단' 발생,국제신문,1일 오전 서울 등 일부 판매점서 발행 중단 복권 운영사 동행복권에 약 50건 민원...,2026-01-02,True,1일 오전 서울 등 일부 판매점서 발행 중단 복권 운영사 동행복권에 약 50건 민원...,"[오전, 서울, 일부, 판매점, 서, 발행, 중단, 복권, 운영, 사, 동, 행복,..."
4,263,"배경훈 ""쿠팡, 5개월치 홈피 접속로그 삭제 방치…법 위반""",연합뉴스TV,'쿠팡 사태 범정부 TF' 팀장인 배경훈 부총리 겸 과기정통부 장관은 쿠팡 측의 과...,2026-01-02,True,쿠팡 사태 범정부 TF 팀장인 배경훈 부총리 겸 과기정통부 장관은 쿠팡 측의 과실...,"[쿠팡, 사태, 정부, 팀장, 배, 경, 후, 부총리, 과기, 정통부, 장관, 쿠팡..."


In [37]:
#   answer 열 빈도 확인
df_tokenizing['answer'].value_counts()

answer
True     328331
False    280475
Name: count, dtype: int64

In [38]:
#   입력 데이터 및 정답 데이터 추출
article_list = list(df_tokenizing['tokens'])
answer_list = list(df_tokenizing['answer'])

len(article_list),len(answer_list)

(608806, 608806)

In [39]:
# 학습 데이터 및 테스트 데이터로 분리
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(article_list, answer_list, test_size = 0.2, random_state = 42, stratify = answer_list)

len(X_train), len(X_test), len(y_train), len(y_test)

(487044, 121762, 487044, 121762)

### 6. 학습 데이터 준비

### 1) tokenizer 생성(용도: Integer Encoding)

In [40]:
#   Tokenizer 생성: 분석 단어수 확인
"""num_words 지정 경우, 사용할 단어 수(vocab_size) + 1 (0은 OOV에 할당)""" 
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer_001 = Tokenizer()
tokenizer_001.fit_on_texts(X_train)

In [41]:
#   단어 수 확인
len(tokenizer_001.word_index)

267561

In [42]:
from 감성분석_260523_analyzer_add_cnt_word import cnt_word

#   등장빈도 수 설정
threshold = 25

#   threshold 이하 단어 등장 갯수 및 등장 빈도 확인
cnt_word(tokenizer_001.word_counts.items(), threshold)

전체 단어 :  267,561개  105,826,133번 등장
제외 단어 (등장빈도 25이하) :  222,221개  933,008번 등장
제외 단어 비율 : 단어 빈도 83.05%/등장 빈도 0.88%
제외 단어 반영 : 45,340개 99.12%


In [43]:
#   Tokenizer 생성: 단어 수 제한 적용
#   기준: 단어 출현 빈도 26이상 단어(45,340개)
num_vocab = 45340
num_words = num_vocab + 1#  + 1: index 0은 padding 용도로 예약

tokenizer_001 = Tokenizer(num_words = num_words)
tokenizer_001.fit_on_texts(X_train)


In [44]:
#   단어 수 확인
len(tokenizer_001.word_index), tokenizer_001.num_words

(267561, 45341)

### 2) 입력 데이터 Integer Encoding

In [45]:
"""
제한된 단어에만 index 부여
희귀 단어로만 구성된 review: 단어가 0 → 결측치에 해당(제거必)
"""

#   입력 데이터 Integer Encoding
encoded_X_train = tokenizer_001.texts_to_sequences(X_train)

In [46]:
#   Integer Encoding 결과 확인
print(len(encoded_X_train))
print(encoded_X_train[1])

487044
[1835, 773, 280, 27, 151, 55, 1, 280, 300, 696, 27, 268, 2926, 690, 21, 58, 280, 300, 696, 280, 202, 27, 708, 696, 2007, 138, 799, 2654, 696, 7, 27, 151, 3306, 639, 377, 1, 62, 21995, 329, 106, 528, 2206, 280, 300, 696, 44, 1282, 3811, 46, 31, 20, 1616, 374, 799, 62, 1766, 357, 244, 280, 55, 991, 12, 155, 9, 2206, 30, 67, 696, 224, 10883, 23073, 309, 15, 2, 300, 696, 62, 1529, 44, 196, 1440, 219, 2206, 150, 62, 44, 4439, 3712, 1734, 2418, 1835, 219, 231, 357, 290, 148, 175, 696, 2007, 1808, 310, 1447, 1209, 556, 210, 119, 87, 314, 86, 41, 1440, 219, 195, 11, 62, 513, 27, 508, 300, 696, 2007, 2206, 4439, 137, 513, 62, 357, 160, 32, 11, 2994, 809, 743, 696, 2007, 1808, 310, 1115, 708, 195, 11, 62, 513, 300, 696, 44, 20, 6, 111, 255, 333, 523, 1812, 329, 104, 280, 202, 27, 799, 696, 2007, 10, 790, 350, 498, 15, 299, 1143, 1669, 95, 3008, 300, 696, 433, 55, 173, 434, 13, 148, 82, 37, 30, 25, 21, 58, 280, 202, 27, 2, 603, 82, 778, 189, 202, 100, 638, 12, 202, 42, 1452, 55, 350, 6, 30

In [47]:
print(encoded_X_train[1])

[1835, 773, 280, 27, 151, 55, 1, 280, 300, 696, 27, 268, 2926, 690, 21, 58, 280, 300, 696, 280, 202, 27, 708, 696, 2007, 138, 799, 2654, 696, 7, 27, 151, 3306, 639, 377, 1, 62, 21995, 329, 106, 528, 2206, 280, 300, 696, 44, 1282, 3811, 46, 31, 20, 1616, 374, 799, 62, 1766, 357, 244, 280, 55, 991, 12, 155, 9, 2206, 30, 67, 696, 224, 10883, 23073, 309, 15, 2, 300, 696, 62, 1529, 44, 196, 1440, 219, 2206, 150, 62, 44, 4439, 3712, 1734, 2418, 1835, 219, 231, 357, 290, 148, 175, 696, 2007, 1808, 310, 1447, 1209, 556, 210, 119, 87, 314, 86, 41, 1440, 219, 195, 11, 62, 513, 27, 508, 300, 696, 2007, 2206, 4439, 137, 513, 62, 357, 160, 32, 11, 2994, 809, 743, 696, 2007, 1808, 310, 1115, 708, 195, 11, 62, 513, 300, 696, 44, 20, 6, 111, 255, 333, 523, 1812, 329, 104, 280, 202, 27, 799, 696, 2007, 10, 790, 350, 498, 15, 299, 1143, 1669, 95, 3008, 300, 696, 433, 55, 173, 434, 13, 148, 82, 37, 30, 25, 21, 58, 280, 202, 27, 2, 603, 82, 778, 189, 202, 100, 638, 12, 202, 42, 1452, 55, 350, 6, 30, 893, 

In [48]:
#   길이 0 index 추출
#   X_train 해당 index 추출 → y_train에서도 삭제
#   enumerate: 튜플 해제
null_index = [index for index, article in enumerate(encoded_X_train) if len(article) < 1]
len(null_index)

1150

In [49]:
#   길이 0 기사 → 재구성(길이 1)
new_X_train = [article for index, article in enumerate(encoded_X_train) if index not in null_index]
new_y_train = [label for index, label in enumerate(y_train) if index not in null_index]

In [50]:
#   재구성 데이터 생성 확인
len(new_X_train), len(new_y_train)

(485894, 485894)

### 3) 입력 데이터 padding
* 입력 데이터의 길이(max_len)를 정하여 padding

In [51]:
#   기사 길이 분포 확인
len_df = pd.DataFrame([len(article) for article in new_X_train])

In [52]:
#   기술통계 확인
len_df.describe()

,0
count,485894.000000
mean,215.876559
std,137.790193
min,1.000000
25%,131.000000
50%,193.000000
75%,275.000000
max,3802.000000


In [53]:
from mylib.my_utils import below_threshold_len_from_list
#   길이 max_len 이하 데이터 비중 확인
max_len = 700

below_threshold_len_from_list(max_len, new_X_train)

길이가 700 이하인 text의 비율 : 99.22%


In [54]:
#   데이터 padding(기준: max_len 길이)
from tensorflow.keras.preprocessing.sequence import pad_sequences

#   신경망 입력 데이터 준비
input_X_train = pad_sequences(new_X_train, maxlen = max_len)

len(input_X_train), input_X_train[:2]

(485894,
 array([[   0,    0,    0, ..., 1183,   35,    9],
        [   0,    0,    0, ...,  294,  699,  542]]))

### 4) 정답 데이터 one-hot encoding

In [55]:
from tensorflow.keras.utils import to_categorical

input_y_train = to_categorical(new_y_train)
len(input_y_train), input_y_train[:2]

(485894,
 array([[1., 0.],
        [1., 0.]], dtype=float32))

## 7. 테스트 데이터 준비

    입력 데이터 Integer Encoding (결측치 제거)
    입력 데이터 padding
    정답 데이터 ont-hot encoding


In [56]:
#   입력 데이터 Integer Encoding
encoded_X_test = tokenizer_001.texts_to_sequences(X_test)
print(len(encoded_X_test), encoded_X_test[:2])

121762 [[162, 1968, 307, 458, 352, 39, 1231, 4925, 516, 110, 1034, 332, 269, 110, 282, 2632, 621, 458, 9, 4925, 52, 581, 41, 458, 193, 66, 76, 132, 162, 110, 270, 270, 458, 440, 131, 1074, 347, 52, 1255, 40, 499, 554, 50, 136, 638, 12, 2, 1066, 955, 1007, 84, 12, 39, 84, 63, 326, 31, 52, 122, 374, 161, 39, 192, 193, 108, 343, 6631, 518, 1074, 200, 307, 64, 10, 100, 252, 4695, 1, 92, 1616, 52, 12, 579, 704, 4925, 110, 49, 548, 156, 511, 345, 146, 179, 68, 238, 44, 592, 153, 102, 1634, 2132, 731, 375, 305, 3360, 193, 548, 156, 118, 42, 517, 165, 2, 246, 905, 120, 2171, 53, 1634, 110, 93, 69, 1, 16, 4925, 78, 2895, 2, 91, 307, 27, 151, 1634, 110, 2632, 653, 516, 110, 2116, 153, 3, 1634, 110, 100, 12, 2830, 151, 138, 562, 565, 52, 145, 52, 839, 110, 49, 275, 264, 50, 110, 241, 548, 142, 505, 890, 2, 246, 810, 120, 35, 9], [871, 1426, 4515, 800, 186, 26, 1217, 485, 142, 635, 901, 871, 1426, 4515, 800, 186, 26, 800, 2501, 1, 14479, 4076, 245, 513, 44, 1217, 485, 142, 635, 14479, 1, 1426, 211

In [57]:
#   길이가 0인 기사 index 추출(있으면 길이 1 이상인 기사만으로 테스트 데이터 구성)
null_index = [index for index, article in enumerate(encoded_X_test) if len(article) < 1]
len(null_index)

284

In [58]:
new_X_test = [article for index, article in enumerate(encoded_X_test) if index not in null_index]
new_y_test = [label for index, label in enumerate(y_test) if index not in null_index]

len(new_X_test), len(new_y_test)

(121478, 121478)

In [59]:
#   입력 데이터 padding
input_X_test = pad_sequences(new_X_test, maxlen = max_len)
len(input_X_test), input_X_test[:2]

(121478,
 array([[  0,   0,   0, ..., 120,  35,   9],
        [  0,   0,   0, ..., 169,  21,  19]]))

In [60]:
#   정답 데이터 ont-hot encoding
input_y_test = to_categorical(new_y_test)
len(input_y_test), input_y_test[:2]

(121478,
 array([[0., 1.],
        [0., 1.]], dtype=float32))

## 8. 모델 구축 및 컴파일

In [61]:
#   모델 설계
from tensorflow.keras.layers import Embedding, LSTM, Dense
#   Embedding layer 파라미터
num_features = num_words
embedding_dim = 32# 입력 차원
input_length = max_len

run_nodes = 32# 은닉 노드 수

dense_nodes = 16
output_nodes = 2#   출력 층

rnn_model = [
    Embedding(num_features, embedding_dim, input_length=input_length),
    LSTM(run_nodes),
    Dense(dense_nodes, activation = 'tanh'),
    Dense(output_nodes, activation = 'softmax')
]

In [62]:
#   모델 생성
from tensorflow.keras.models import Sequential
model = Sequential(rnn_model)
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_1 (Embedding)     (None, 700, 32)           1450912   
                                                                 
 lstm_1 (LSTM)               (None, 32)                8320      
                                                                 
 dense_2 (Dense)             (None, 16)                528       
                                                                 
 dense_3 (Dense)             (None, 2)                 34        
                                                                 
Total params: 1,459,794
Trainable params: 1,459,794
Non-trainable params: 0
_________________________________________________________________


In [63]:
#   학습 설계 : 생성한 모델에 학습 관련 하이퍼파라미터 설정
from tensorflow.keras.optimizers import RMSprop

model.compile(loss = 'binary_crossentropy', metrics = 'accuracy', optimizer=RMSprop(learning_rate=0.001))

In [64]:
#   EarlyStopping, Modelcheckpoint callback 함수 설정
#       monitor = 검증데이터 손식, model = 최소화가 되도록, patience = 2번까지 참고
#       verbose = 1: 학습과정 확인
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
es = EarlyStopping(monitor = 'val_loss', mode = 'min', patience = 3, verbose = 1)
#   keras 특유 파일 포맷
checkpoint_filepath = r'./model/best_model_article.h5'
#   save_best_only: 하이퍼파라미터 설정 적용 때마다s
mc = ModelCheckpoint(checkpoint_filepath, monitor = 'val_loss', mode = 'min', save_best_only = True)

##  9. 모델 학습

In [44]:
#   모델 학습
model.fit(input_X_train, input_y_train, epochs = 20, batch_size = 128, validation_split = 0.1, callbacks=[es, mc])

Epoch 1/20
3417/3417 [==============================] - 554s 162ms/step - loss: 0.6358 - accuracy: 0.6349 - val_loss: 0.6168 - val_accuracy: 0.6628
Epoch 2/20
3417/3417 [==============================] - 645s 189ms/step - loss: 0.5893 - accuracy: 0.6856 - val_loss: 0.6005 - val_accuracy: 0.6783
Epoch 3/20
3417/3417 [==============================] - 652s 191ms/step - loss: 0.5562 - accuracy: 0.7127 - val_loss: 0.5635 - val_accuracy: 0.7055
Epoch 4/20
3417/3417 [==============================] - 634s 185ms/step - loss: 0.5259 - accuracy: 0.7358 - val_loss: 0.6080 - val_accuracy: 0.6676
Epoch 5/20
3417/3417 [==============================] - 852s 249ms/step - loss: 0.4988 - accuracy: 0.7554 - val_loss: 0.5484 - val_accuracy: 0.7086
Epoch 6/20
3417/3417 [==============================] - 1112s 325ms/step - loss: 0.4710 - accuracy: 0.7741 - val_loss: 0.5430 - val_accuracy: 0.7221
Epoch 7/20
3417/3417 [==============================] - 553s 162ms/step - loss: 0.4449 - accuracy: 0.7901 - val

In [47]:
#   학습 결과 저장
model.save('./model/model_final_article.h5')

In [70]:
import numpy as np

#   입력 데이터 저장
np.save('./data/article_X_test.npy', input_X_test)
np.save('./data/article_y_test.npy', input_y_test)